In [1]:
%env DATA_PATH=../../../data
from lib.fit import load_fit_file, get_gps_data, get_camera_starts, get_camera_ends, get_sensor_data
import json
import pandas as pd
import os
from tqdm.notebook import tqdm
import plotly.express as px
import subprocess

DATA_PATH = '../../../data'

env: DATA_PATH=../../../data


In [2]:
fit = load_fit_file(f"{DATA_PATH}/archive/tmp/calibrate/2026-05-03-23-18-49.fit")

In [83]:
# s = stabalization, lc = lens correction
idx_no_s = 2
idx_no_s_lc = 3
idx = idx_no_s_lc
camera_starts = get_camera_starts(fit)
camera_ends = get_camera_ends(fit)
start, end = camera_starts[idx], camera_ends[idx]
bag_folder = f"{DATA_PATH}/archive/tmp/calibrate/bag_data{idx}"
os.makedirs(bag_folder, exist_ok=True)

## IMU

In [84]:
calibration_mesgs = fit['three_d_sensor_calibration_mesgs']
calibration_data = { m['sensor_type']: m for m in calibration_mesgs }

In [85]:
accel_cal = calibration_data['accelerometer']
accel_raw, accel_data, fs = get_sensor_data(accel_cal, fit['accelerometer_data_mesgs'], {'alpha_x': 'accel_x', 'alpha_y': 'accel_y', 'alpha_z': 'accel_z'})
accel = accel_data.loc[start:end].drop(columns=['timestamp'])
accel

,alpha_x,alpha_y,alpha_z
timestamp,,,
564264,0.067383,0.503906,-0.849609
564274,0.062988,0.501953,-0.841797
564284,0.054199,0.484375,-0.829102
564294,0.052246,0.464355,-0.816895
564304,0.057129,0.466797,-0.819824
...,...,...,...
690013,-0.020508,0.236816,-0.969727
690023,-0.006348,0.253418,-0.943848
690033,0.003418,0.246582,-0.918945


In [86]:
gyro_cal = calibration_data['gyroscope']
gyro_raw, gyro_data, gyro_fs = get_sensor_data(gyro_cal, fit['gyroscope_data_mesgs'], {'omega_x': 'gyro_x', 'omega_y': 'gyro_y', 'omega_z': 'gyro_z'})
gyro = gyro_data.loc[start:end].drop(columns=['timestamp'])
gyro

,omega_x,omega_y,omega_z
timestamp,,,
564264,-2.987805,7.987805,9.024390
564274,-6.280488,10.426829,7.195122
564284,-7.256098,10.426829,7.378049
564294,-5.609756,10.792683,9.146341
564304,-3.414634,10.609756,9.695122
...,...,...,...
690013,-2.134146,-1.707317,4.146341
690023,-2.378049,1.219512,2.682927
690033,-0.426829,3.292683,2.195122


In [87]:
imu = pd.merge_asof(accel, gyro, on='timestamp')
imu.timestamp = (imu.timestamp * 1e6).astype('int64')
imu.to_csv(f"{bag_folder}/imu0.csv", index=False, header=True)

In [6]:
with open(f"{DATA_PATH}/archive/tmp/calibrate/fit.json", 'w') as f:
    json.dump(fit, f, default=str, indent=2)

## Video

In [76]:
import cv2
videos = ['VIRB0119.MP4', 'VIRB0120.MP4', 'VIRB0121.MP4', 'VIRB0122.MP4']
p = f"{DATA_PATH}/archive/tmp/calibrate/{videos[idx]}"

In [ ]:
cap = cv2.VideoCapture(p)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {p}")

frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
progress = tqdm(total=frame_count, desc="Saving frames")

saved_frames = 0
try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        timestamp_ms = start + cap.get(cv2.CAP_PROP_POS_MSEC)
        timestamp_ns = int(round(timestamp_ms * 1_000_000))
        out_path = os.path.join(bag_folder, f"cam0/{timestamp_ns}.png")
        cv2.imwrite(out_path, frame)
        saved_frames += 1
        progress.update(1)
finally:
    progress.close()
    cap.release()

saved_frames

Saving frames:   0%|          | 0/3777 [00:00<?, ?it/s]

3777

In [50]:
((end - start) / 1000) * 60

7558.5

## IMU Noise

In [2]:
fit = load_fit_file(f"{DATA_PATH}/archive/tmp/calibrate/2026-05-07-19-44-00.fit")

Caching archive_tmp_calibrate_2026-05-07-19-44-00.json


In [23]:
calibration_mesgs = fit['three_d_sensor_calibration_mesgs']
calibration_data = { m['sensor_type']: m for m in calibration_mesgs }
start, end = 4_000_000, 15_000_000

In [17]:
accel_cal = calibration_data['accelerometer']
accel_raw, accel_data, fs = get_sensor_data(accel_cal, fit['accelerometer_data_mesgs'], {'alpha_x': 'accel_x', 'alpha_y': 'accel_y', 'alpha_z': 'accel_z'})
accel = accel_data.drop(columns=['timestamp']).loc[start:end]
accel

,alpha_x,alpha_y,alpha_z
timestamp,,,
4000006,0.080078,0.935547,-0.047363
4000016,0.080566,0.934570,-0.047852
4000026,0.081543,0.935059,-0.046875
4000036,0.082031,0.937500,-0.045898
4000046,0.081543,0.938965,-0.045898
...,...,...,...
14999957,0.081543,0.935547,-0.046387
14999967,0.082520,0.935547,-0.046387
14999977,0.082031,0.936523,-0.046387


In [18]:
gyro_cal = calibration_data['gyroscope']
gyro_raw, gyro_data, gyro_fs = get_sensor_data(gyro_cal, fit['gyroscope_data_mesgs'], {'omega_x': 'gyro_x', 'omega_y': 'gyro_y', 'omega_z': 'gyro_z'})
gyro = gyro_data.drop(columns=['timestamp']).loc[start:end]
gyro

,omega_x,omega_y,omega_z
timestamp,,,
4000006,-0.121951,-0.975610,0.731707
4000016,-0.060976,-0.975610,0.731707
4000026,-0.060976,-0.914634,0.731707
4000036,-0.060976,-0.975610,0.792683
4000046,-0.121951,-0.914634,0.853659
...,...,...,...
14999957,0.000000,-0.853659,0.853659
14999967,0.000000,-0.914634,0.853659
14999977,0.060976,-0.975610,0.853659


In [20]:
imu = pd.merge_asof(accel, gyro, on='timestamp')
imu.timestamp = (imu.timestamp * 1e6).astype('int64')
imu.to_csv(f"{DATA_PATH}/archive/tmp/calibrate/noise_bag/imu0.csv", index=False, header=True)

In [24]:
(end - start) / 1000

11000.0